# Analyzing Files

In this notebook, we send PDF files to an OpenAI model and ask whether its an invoice. If that's the case, we extract fields like the vendor and the payment amount from it. 

Docs:
- [Analyze images and files](https://developers.openai.com/api/docs/quickstart?input-type=image-url#analyze-images-and-files)
- [File inputs](https://developers.openai.com/api/docs/guides/file-inputs)

## Setup

In [ ]:
import base64
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel

load_dotenv(override=True)
client = OpenAI()

## The Invoice we are going to process

In [ ]:
INVOICE_PATH = Path("files/Anthropic_Invoice.pdf")

## Defining the model

[GPT-5.4-mini](https://developers.openai.com/api/docs/models/gpt-5.4-mini)

Event though models don't undersand PDFs, we can still send them to the OPEN AI APIs, where text and images from the PDFs are extracted and sent to the model. 

[Full list of supported file types](https://developers.openai.com/api/docs/guides/file-inputs#full-list-of-accepted-file-types)

Important: For text-based files such as `.docx` or `.pptx`, the API extracts only the text. Images, charts, and other visual content inside those files are not "seen" by the model. Convert these files to PDF locally if you want their visual content to be processed.

In [ ]:
MODEL = "gpt-5.4-mini"

## Define Response Format

A Pydantic model gives the result a clear shape. The model first classifies whether the file is really an invoice. The invoice fields can be `None` when the file is not an invoice or when a value is missing.

In [ ]:
class DocumentAnalysis(BaseModel):
    is_invoice: bool
    vendor: str | None
    payment_date: str | None
    amount: float | None
    currency: str | None

# Define Instructions

In [ ]:
INSTRUCTIONS = (
    "Decide whether the document is really an invoice. "
    "If it is not an invoice, set is_invoice to false and set "
    "vendor, payment_date, amount, and currency to null. "
    "If it is an invoice, set is_invoice to true and extract "
    "the invoice vendor, payment date, total amount, and currency. "
    "Use YYYY-MM-DD for the payment date. "
    "Use null for any missing invoice field."
)

## Option 1 - Web URL

In [ ]:
response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_url": "https://raw.githubusercontent.com/LukasLechnerDev/AI-Engineering-Foundations-Labs/main/3-interacting-with-llm-apis/files/Anthropic_Invoice.pdf",
                },
            ],
        },
    ],
    text_format=DocumentAnalysis,
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

## Option 2 - Send base64 encoded file

The base64 encoded PDF is sent as an `input_file`. The extraction rules go into `instructions`, because they are stable app instructions.

In [ ]:
invoice_base64 = base64.b64encode(INVOICE_PATH.read_bytes()).decode("utf-8")

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": INVOICE_PATH.name,
                    "file_data": f"data:application/pdf;base64,{invoice_base64}",
                },
            ],
        }
    ],
    text_format=DocumentAnalysis,
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

## Option 3 - File Upload

In case you need to make multiple calls for a file, you can upload files first, and then reference them with the file id. 

`client.files.create(...)` reads the local file, uploads it to OpenAI, and returns a file object with an id you can use later. 

`"rb"` means read binary, so the PDF is opened as raw bytes for uploading.

In [ ]:
print("Start uploading the file to the OpenAI API...")
file = client.files.create(
    file=open(INVOICE_PATH, "rb"),
    purpose="user_data",
)

print("Upload completed. File ID:", file.id)

response = client.responses.parse(
    model=MODEL,
    instructions=INSTRUCTIONS,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
            ],
        },
    ],
    text_format=DocumentAnalysis,
)

document_analysis = response.output_parsed

if document_analysis is not None:
    print(document_analysis.model_dump_json(indent=2))

# Further processing
After we have extracted the most important invoice data, we can do some additional steps, like: 

- Rename each PDF using a consistent pattern, such as `YYYY_MM_DD-Company_Name.pdf` (`2026_05_09-Anthropic_Ireland_Limited.pdf`), then upload it to the correct Google Drive folder.
- Save the extracted fields in a CSV file, spreadsheet, or database so invoices can be searched and filtered later.
- Import the data into your bookkeeping software